# Day 7 Lab 09: Dimension Enrichment

        **Layer:** Silver  
        **Python reference:** `Lab_Files/labs/lab_09_dimension_enrichment.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Read current orders from Silver.
- Join customer, product, and FX dimensions through the shared helper.
- Inspect match status and normalized USD values.

**Dependency:** Run Lab/Notebook 08 first so `lake/silver/orders_current` exists.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [1]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


Working directory: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files
Using lab helpers from: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files\labs


## 1. Import Lab Helpers

In [2]:

import importlib
import day7_common
day7_common = importlib.reload(day7_common)

from day7_common import LAKE_DIR, OUTPUT_DIR, enriched_orders, ensure_output_dirs, require_source_data, spark_session, write_csv_dir, read_parquet, write_parquet

## 2. Start Spark and Read Current Orders

In [3]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook09DimensionEnrichment")

current = read_parquet(spark, LAKE_DIR / "silver" / "orders_current")
current.select("order_id", "customer_id", "product_id", "currency", "amount").show(20, truncate=False)

+--------+-----------+----------+--------+-------+
|order_id|customer_id|product_id|currency|amount |
+--------+-----------+----------+--------+-------+
|1001    |501        |P-LAP-01  |USD     |1299.99|
|1002    |502        |P-PHN-02  |USD     |799.0  |
|1003    |503        |P-HDP-03  |USD     |149.5  |
|1004    |504        |P-CAM-04  |USD     |450.0  |
|1006    |504        |P-BAG-05  |EUR     |89.99  |
|1007    |999        |P-CAM-04  |USD     |450.0  |
|1009    |505        |P-SVC-99  |USD     |40.0   |
+--------+-----------+----------+--------+-------+



## 3. Apply Dimension Enrichment

In [4]:
enriched = enriched_orders(spark, current)

## 4. Inspect Enriched Rows

In [5]:
preview = enriched.select(
    "order_id",
    "customer_name",
    "region",
    "product_name",
    "product_category",
    "status",
    "amount_usd",
    "customer_match_status",
).orderBy("order_id")
preview.show(20, truncate=False)

+--------+-------------+-------+-----------------------+----------------+---------+----------+---------------------+
|order_id|customer_name|region |product_name           |product_category|status   |amount_usd|customer_match_status|
+--------+-------------+-------+-----------------------+----------------+---------+----------+---------------------+
|1001    |Ada Retail   |NA     |Cloud Laptop           |ELECTRONICS     |SHIPPED  |1299.99   |MATCHED              |
|1002    |Ravi Stores  |APAC   |Edge Phone             |ELECTRONICS     |CANCELLED|799.0     |MATCHED              |
|1003    |Sofia Market |EMEA   |Noise Cancel Headphones|ACCESSORIES     |NEW      |149.5     |MATCHED              |
|1004    |Mina Travel  |EMEA   |Action Camera          |ELECTRONICS     |CANCELLED|450.0     |MATCHED              |
|1006    |Mina Travel  |EMEA   |Travel Bag             |TRAVEL          |NEW      |97.19     |MATCHED              |
|1007    |null         |UNKNOWN|Action Camera          |ELECTRON

## 5. Write Enriched Silver Table

In [6]:
enriched_path = LAKE_DIR / "silver" / "orders_enriched"
write_parquet(enriched, enriched_path, mode="overwrite")
write_csv_dir(preview, OUTPUT_DIR / "notebook_09_enriched_orders_preview")
print(f"Enriched current orders: {enriched.count()}")

Enriched current orders: 7


## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [7]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()